In [9]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. RUTAS DEFINITIVAS
# =========================================================
# Carpeta con todos los CSV que quieres convertir en tablas
CSV_FOLDER = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\tablas en csv"
# Base de datos destino
DB_PATH = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\spotify.db"
# Carpeta de salida para gráficos e informes
OUTPUT_DIR = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)

# =========================================================
# 2. IMPORTAR CADA CSV A SQLITE
# =========================================================
conn = sqlite3.connect(DB_PATH)

print(f"Buscando archivos CSV en: {CSV_FOLDER}")
for filename in os.listdir(CSV_FOLDER):
    if filename.endswith(".csv"):
        file_path = os.path.join(CSV_FOLDER, filename)
        # Nombre de tabla basado en el nombre del archivo (sin .csv)
        table_name = os.path.splitext(filename)[0].replace(" ", "_")
        
        df_temp = pd.read_csv(file_path)
        df_temp.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"✅ Tabla '{table_name}' creada desde {filename}")

# =========================================================
# 3. VISUALIZACIÓN Y REPORTE (ANALIZANDO LAS TABLAS)
# =========================================================
# Aquí usamos las tablas que acabamos de crear en SQLite

# A) Ejemplo de gráfico desde una de las tablas importadas
# Sustituye 'albums' por una tabla real que haya aparecido arriba
tablas_disponibles = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist()
print(f"\nTablas disponibles para graficar: {tablas_disponibles}")

if 'albums' in tablas_disponibles:
    df_albums = pd.read_sql("SELECT artists, COUNT(*) as total FROM albums GROUP BY artists ORDER BY total DESC LIMIT 15", conn)
    plt.figure()
    sns.barplot(data=df_albums, x="total", y="artists", palette="magma")
    plt.title("Top 15 Artistas (Datos reales de 'albums')")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "viz_artistas.png"))
    plt.close()

# B) Ejemplo de gráfico desde 'genre'
if 'genre' in tablas_disponibles:
    df_genre = pd.read_sql("SELECT track_genre, COUNT(*) as total FROM genre GROUP BY track_genre ORDER BY total DESC LIMIT 15", conn)
    plt.figure()
    sns.barplot(data=df_genre, x="total", y="track_genre", palette="viridis")
    plt.title("Top 15 Géneros (Datos reales de 'genre')")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "viz_generos.png"))
    plt.close()

# =========================================================
# 4. COMPILAR INFORME
# =========================================================
report = f"""# Informe de Análisis Spotify
- **Total de tablas cargadas:** {len(tablas_disponibles)}
- **Tablas:** {', '.join(tablas_disponibles)}

## Análisis
- Se han procesado los archivos CSV reales de la ruta proporcionada.
- Las visualizaciones generadas reflejan la distribución de artistas y géneros obtenida mediante consultas SQL directas sobre las tablas normalizadas.
"""

with open(os.path.join(OUTPUT_DIR, "reporte_final.md"), "w", encoding="utf-8") as f:
    f.write(report)

conn.close()
print(f"\n🎉 Proceso finalizado. Gráficos y reporte guardados en: {OUTPUT_DIR}")

Buscando archivos CSV en: C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\tablas en csv
✅ Tabla 'albums' creada desde albums.csv
✅ Tabla 'Audio_Analysis' creada desde Audio_Analysis.csv
✅ Tabla 'Audio_Features' creada desde Audio_Features.csv
✅ Tabla 'genre' creada desde genre.csv

Tablas disponibles para graficar: ['albums', 'Audio_Analysis', 'Audio_Features', 'genre']


C:\Users\juanb\AppData\Local\Temp\ipykernel_16256\2675824715.py:50: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_albums, x="total", y="artists", palette="magma")
C:\Users\juanb\AppData\Local\Temp\ipykernel_16256\2675824715.py:60: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_genre, x="total", y="track_genre", palette="viridis")



🎉 Proceso finalizado. Gráficos y reporte guardados en: C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\output


In [10]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. RUTAS
# =========================================================
DB_PATH = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\spotify.db"
OUTPUT_DIR = r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database\output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 2. CONFIGURACIÓN VISUAL
# =========================================================
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# =========================================================
# 3. CONEXIÓN
# =========================================================
conn = sqlite3.connect(DB_PATH)

# =========================================================
# 4. CONSULTAS REALES POR TABLA
# =========================================================

# albums -> top artistas
q_albums = """
SELECT artists, COUNT(*) AS total_tracks
FROM albums
WHERE artists IS NOT NULL AND TRIM(artists) <> ''
GROUP BY artists
ORDER BY total_tracks DESC
LIMIT 15;
"""
df_albums = pd.read_sql_query(q_albums, conn)

# genre -> top géneros
q_genre = """
SELECT track_genre, COUNT(*) AS total_tracks
FROM genre
WHERE track_genre IS NOT NULL AND TRIM(track_genre) <> ''
GROUP BY track_genre
ORDER BY total_tracks DESC
LIMIT 15;
"""
df_genre = pd.read_sql_query(q_genre, conn)

# Audio_Analysis -> categorías de tempo
q_tempo = """
SELECT tempo_category, COUNT(*) AS total_tracks
FROM Audio_Analysis
GROUP BY tempo_category
ORDER BY total_tracks DESC;
"""
df_tempo = pd.read_sql_query(q_tempo, conn)

# Audio_Analysis -> canciones con energía alta vs no alta
q_energy = """
SELECT
    CASE
        WHEN is_energy_high = 1 THEN 'Alta'
        ELSE 'Media/Baja'
    END AS energy_level,
    COUNT(*) AS total_tracks
FROM Audio_Analysis
GROUP BY energy_level
ORDER BY total_tracks DESC;
"""
df_energy = pd.read_sql_query(q_energy, conn)

# Audio_Features -> promedio de métricas principales
q_features = """
SELECT
    AVG(danceability) AS danceability,
    AVG(energy) AS energy,
    AVG(acousticness) AS acousticness,
    AVG(valence) AS valence,
    AVG(liveness) AS liveness,
    AVG(speechiness) AS speechiness
FROM Audio_Features;
"""
df_features = pd.read_sql_query(q_features, conn)
df_features_long = df_features.melt(var_name="feature", value_name="avg_value")

# join real entre tablas -> popularidad media por género
q_pop_genre = """
SELECT
    g.track_genre,
    ROUND(AVG(a.popularity), 2) AS avg_popularity,
    COUNT(*) AS total_tracks
FROM genre g
JOIN albums a
    ON g.track_id = a.track_id
WHERE g.track_genre IS NOT NULL AND TRIM(g.track_genre) <> ''
GROUP BY g.track_genre
HAVING COUNT(*) >= 10
ORDER BY avg_popularity DESC
LIMIT 15;
"""
df_pop_genre = pd.read_sql_query(q_pop_genre, conn)

# =========================================================
# 5. GRÁFICOS DE BARRAS
# =========================================================

# 1) albums
if not df_albums.empty:
    plt.figure()
    sns.barplot(
        data=df_albums,
        x="total_tracks",
        y="artists",
        hue="artists",
        dodge=False,
        legend=False,
        palette="magma"
    )
    plt.title("Top 15 artistas con más canciones")
    plt.xlabel("Número de canciones")
    plt.ylabel("Artista")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "01_albums_top_artistas.png"), dpi=300, bbox_inches="tight")
    plt.close()

# 2) genre
if not df_genre.empty:
    plt.figure()
    sns.barplot(
        data=df_genre,
        x="total_tracks",
        y="track_genre",
        hue="track_genre",
        dodge=False,
        legend=False,
        palette="viridis"
    )
    plt.title("Top 15 géneros con más canciones")
    plt.xlabel("Número de canciones")
    plt.ylabel("Género")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_genre_top_generos.png"), dpi=300, bbox_inches="tight")
    plt.close()

# 3) Audio_Analysis tempo
if not df_tempo.empty:
    plt.figure()
    sns.barplot(
        data=df_tempo,
        x="tempo_category",
        y="total_tracks",
        hue="tempo_category",
        dodge=False,
        legend=False,
        palette="coolwarm"
    )
    plt.title("Distribución de canciones por categoría de tempo")
    plt.xlabel("Categoría de tempo")
    plt.ylabel("Número de canciones")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "03_audio_analysis_tempo.png"), dpi=300, bbox_inches="tight")
    plt.close()

# 4) Audio_Analysis energy
if not df_energy.empty:
    plt.figure()
    sns.barplot(
        data=df_energy,
        x="energy_level",
        y="total_tracks",
        hue="energy_level",
        dodge=False,
        legend=False,
        palette="rocket"
    )
    plt.title("Canciones con energía alta vs media/baja")
    plt.xlabel("Nivel de energía")
    plt.ylabel("Número de canciones")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "04_audio_analysis_energy.png"), dpi=300, bbox_inches="tight")
    plt.close()

# 5) Audio_Features
if not df_features_long.empty:
    plt.figure()
    sns.barplot(
        data=df_features_long,
        x="feature",
        y="avg_value",
        hue="feature",
        dodge=False,
        legend=False,
        palette="Set2"
    )
    plt.title("Promedio de audio features")
    plt.xlabel("Feature")
    plt.ylabel("Valor promedio")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "05_audio_features_promedio.png"), dpi=300, bbox_inches="tight")
    plt.close()

# 6) Join albums + genre
if not df_pop_genre.empty:
    plt.figure()
    sns.barplot(
        data=df_pop_genre,
        x="avg_popularity",
        y="track_genre",
        hue="track_genre",
        dodge=False,
        legend=False,
        palette="cubehelix"
    )
    plt.title("Top 15 géneros por popularidad media")
    plt.xlabel("Popularidad media")
    plt.ylabel("Género")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "06_popularidad_media_genero.png"), dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# 6. INFORME
# =========================================================
report_lines = []
report_lines.append("# Informe de visualización y análisis Spotify\n\n")
report_lines.append("## Tablas utilizadas\n")
report_lines.append("- albums\n")
report_lines.append("- genre\n")
report_lines.append("- Audio_Features\n")
report_lines.append("- Audio_Analysis\n\n")

report_lines.append("## Visualizaciones generadas\n")
report_lines.append("- Top artistas a partir de albums.\n")
report_lines.append("- Top géneros a partir de genre.\n")
report_lines.append("- Distribución por categoría de tempo a partir de Audio_Analysis.\n")
report_lines.append("- Comparación de energía alta vs media/baja desde Audio_Analysis.\n")
report_lines.append("- Promedio de audio features desde Audio_Features.\n")
report_lines.append("- Popularidad media por género mediante join entre albums y genre.\n\n")

report_lines.append("## Insights\n")
if not df_albums.empty:
    report_lines.append(f"- El artista con más canciones es {df_albums.iloc[0]['artists']} con {int(df_albums.iloc[0]['total_tracks'])} registros.\n")
if not df_genre.empty:
    report_lines.append(f"- El género con más canciones es {df_genre.iloc[0]['track_genre']} con {int(df_genre.iloc[0]['total_tracks'])} registros.\n")
if not df_tempo.empty:
    report_lines.append(f"- La categoría de tempo predominante es {df_tempo.iloc[0]['tempo_category']} con {int(df_tempo.iloc[0]['total_tracks'])} canciones.\n")
if not df_pop_genre.empty:
    report_lines.append(f"- El género con mayor popularidad media es {df_pop_genre.iloc[0]['track_genre']} con {df_pop_genre.iloc[0]['avg_popularity']} de popularidad media.\n")

report_lines.append("\n## Conclusión\n")
report_lines.append("Se han desarrollado visualizaciones con bibliotecas de Python y se ha compilado un informe con análisis e insights a partir de las tablas del proyecto.\n")

with open(os.path.join(OUTPUT_DIR, "reporte_visualizaciones.md"), "w", encoding="utf-8") as f:
    f.writelines(report_lines)

conn.close()

print("✅ Gráficos de barras e informe generados correctamente en output/")

✅ Gráficos de barras e informe generados correctamente en output/
